# Hybrid Table Extraction: UniTable + Gemini
**3-way comparison on PubTabNet (20 images)**

| Method | What it does |
|--------|-------------|
| **UniTable** | GPU model extracts HTML structure + OCR (fast, deterministic) |
| **Gemini** | Vision LLM extracts table from scratch (no UniTable) |
| **Hybrid** | UniTable extracts HTML → Gemini fixes OCR errors using image + HTML |

**Setup:** Runtime → Change runtime type → **T4 GPU**, then run all cells.

In [ ]:
# ============================================================
# Cell 1: Install deps, clone UniTable, download weights + dataset
# ============================================================
import os, subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "tokenizers", "Pillow", "pandas",
    "google-genai", "python-dotenv", "beautifulsoup4", "lxml",
    "huggingface_hub"])

# Clone UniTable repo (needed for model architecture source code)
if not os.path.exists("unitable"):
    subprocess.check_call(["git", "clone", "-q", "https://github.com/poloclub/unitable.git"])
    print("Cloned UniTable repo")
else:
    print("UniTable repo already present")

# Download 3 model weights from HuggingFace (~1.5GB total)
from huggingface_hub import hf_hub_download

WEIGHTS_DIR = "unitable/experiments/unitable_weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

for wf in ["unitable_large_structure.pt", "unitable_large_bbox.pt", "unitable_large_content.pt"]:
    dest = os.path.join(WEIGHTS_DIR, wf)
    if not os.path.exists(dest):
        print(f"Downloading {wf}...")
        hf_hub_download(repo_id="poloclub/UniTable", filename=wf, local_dir=WEIGHTS_DIR)
    else:
        print(f"{wf} ready")

# Clone the HumaAI repo to get PubTabNet examples (images + ground truth)
if not os.path.exists("HumaAI"):
    subprocess.check_call(["git", "clone", "-q", "--depth", "1",
        "https://github.com/roshgill/Table-Extraction-from-Medical-Documents.git", "HumaAI"])
    print("Cloned HumaAI repo")
else:
    print("HumaAI repo already present")

EXAMPLES_DIR = "HumaAI/pubtabnet/examples"
print(f"\nPubTabNet images: {len([f for f in os.listdir(EXAMPLES_DIR) if f.endswith('.png')])} PNGs")
print("Setup complete!")

In [ ]:
# ============================================================
# Cell 2: UniTable extractor (self-contained, no external wrapper)
# ============================================================
import re
import warnings
from pathlib import Path
from functools import partial

import torch
from torch import nn, Tensor
import tokenizers as tk
from PIL import Image
from torchvision import transforms

# Add UniTable source to path
sys.path.insert(0, "unitable")
from src.model import EncoderDecoder, ImgLinearBackbone, Encoder, Decoder
from src.utils import (
    subsequent_mask, pred_token_within_range, greedy_sampling,
    bbox_str_to_token_list, cell_str_to_token_list, html_str_to_token_list,
    build_table_from_html_and_cell, html_table_template,
)
from src.trainer.utils import VALID_HTML_TOKEN, VALID_BBOX_TOKEN, INVALID_CELL_TOKEN

warnings.filterwarnings('ignore')


class UniTableExtractor:
    """Extracts HTML tables from images via UniTable's 3-stage pipeline."""

    def __init__(self, model_dir=None, vocab_dir=None, device=None):
        self.d_model, self.patch_size, self.nhead, self.dropout = 768, 16, 12, 0.2

        if model_dir is None:
            model_dir = Path("unitable/experiments/unitable_weights")
        if vocab_dir is None:
            vocab_dir = Path("unitable/vocab")
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model_dir = Path(model_dir)
        self.vocab_dir = Path(vocab_dir)
        self.device = torch.device(device)
        print(f"UniTable using device: {self.device}")

        self.structure_vocab, self.structure_model = self._load(
            self.vocab_dir / "vocab_html.json", 784,
            self.model_dir / "unitable_large_structure.pt")
        self.bbox_vocab, self.bbox_model = self._load(
            self.vocab_dir / "vocab_bbox.json", 1024,
            self.model_dir / "unitable_large_bbox.pt")
        self.content_vocab, self.content_model = self._load(
            self.vocab_dir / "vocab_cell_6k.json", 200,
            self.model_dir / "unitable_large_content.pt")
        print("UniTable loaded!")

    def _load(self, vocab_path, max_seq_len, weights_path):
        vocab = tk.Tokenizer.from_file(str(vocab_path))
        backbone = ImgLinearBackbone(d_model=self.d_model, patch_size=self.patch_size)
        encoder = Encoder(d_model=self.d_model, nhead=self.nhead, dropout=self.dropout,
                          activation="gelu", norm_first=True, nlayer=12, ff_ratio=4)
        decoder = Decoder(d_model=self.d_model, nhead=self.nhead, dropout=self.dropout,
                          activation="gelu", norm_first=True, nlayer=4, ff_ratio=4)
        model = EncoderDecoder(
            backbone=backbone, encoder=encoder, decoder=decoder,
            vocab_size=vocab.get_vocab_size(), d_model=self.d_model,
            padding_idx=vocab.token_to_id("<pad>"), max_seq_len=max_seq_len,
            dropout=self.dropout, norm_layer=partial(nn.LayerNorm, eps=1e-6))
        model.load_state_dict(torch.load(weights_path, map_location="cpu"))
        model = model.to(self.device).eval()
        return vocab, model

    def _to_tensor(self, image, size):
        T = transforms.Compose([
            transforms.Resize(size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.86597056, 0.88463002, 0.87491087],
                std=[0.20686628, 0.18201602, 0.18485524])])
        return T(image).to(self.device).unsqueeze(0)

    def _decode(self, model, img_tensor, prefix, max_len, eos_id, whitelist=None, blacklist=None):
        with torch.no_grad():
            memory = model.encode(img_tensor)
            ctx = torch.tensor(prefix, dtype=torch.int32).repeat(img_tensor.shape[0], 1).to(self.device)
        for _ in range(max_len):
            if all(eos_id in k for k in ctx):
                break
            with torch.no_grad():
                mask = subsequent_mask(ctx.shape[1]).to(self.device)
                logits = model.generator(model.decode(memory, ctx, tgt_mask=mask, tgt_padding_mask=None))[:, -1, :]
            logits = pred_token_within_range(logits.detach(), white_list=whitelist, black_list=blacklist)
            _, next_tok = greedy_sampling(logits)
            ctx = torch.cat([ctx, next_tok], dim=1)
        return ctx

    def extract_html(self, image: Image.Image) -> str:
        """Full 3-stage pipeline: structure → bbox → content → HTML."""
        if image.mode != "RGB":
            image = image.convert("RGB")
        img_size = image.size

        # Stage 1: Structure
        t = self._to_tensor(image, (448, 448))
        pred = self._decode(self.structure_model, t,
            [self.structure_vocab.token_to_id("[html]")], 512,
            self.structure_vocab.token_to_id("<eos>"),
            [self.structure_vocab.token_to_id(i) for i in VALID_HTML_TOKEN])
        pred_html = html_str_to_token_list(
            self.structure_vocab.decode(pred.cpu().numpy()[0], skip_special_tokens=False))

        # Stage 2: Bbox
        t = self._to_tensor(image, (448, 448))
        pred = self._decode(self.bbox_model, t,
            [self.bbox_vocab.token_to_id("[bbox]")], 1024,
            self.bbox_vocab.token_to_id("<eos>"),
            [self.bbox_vocab.token_to_id(i) for i in VALID_BBOX_TOKEN[:449]])
        pred_bbox = bbox_str_to_token_list(
            self.bbox_vocab.decode(pred.cpu().numpy()[0], skip_special_tokens=False))
        # Rescale bboxes to original image size
        ratio = [img_size[0]/448, img_size[1]/448] * 2
        pred_bbox = [[int(round(c*r)) for c, r in zip(b, ratio)] for b in pred_bbox]

        # Stage 3: Content (OCR per cell)
        if pred_bbox:
            cell_tensors = torch.cat([
                self._to_tensor(image.crop(b), (112, 448)) for b in pred_bbox], dim=0)
            pred = self._decode(self.content_model, cell_tensors,
                [self.content_vocab.token_to_id("[cell]")], 200,
                self.content_vocab.token_to_id("<eos>"), blacklist=
                [self.content_vocab.token_to_id(i) for i in INVALID_CELL_TOKEN])
            pred_cell = [cell_str_to_token_list(c) for c in
                self.content_vocab.decode_batch(pred.cpu().numpy(), skip_special_tokens=False)]
            pred_cell = [re.sub(r'(\d).\s+(\d)', r'\1.\2', c) for c in pred_cell]
        else:
            pred_cell = []

        # Combine
        code = build_table_from_html_and_cell(pred_html, pred_cell)
        return html_table_template("".join(code))


# Initialize
extractor = UniTableExtractor()
print(f"Device: {extractor.device}")

In [ ]:
# ============================================================
# Cell 3: Gemini client + refinement prompt
# ============================================================
import google.genai as genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-2.0-flash"

REFINE_PROMPT = """You are a table data cleanup agent. You receive:
1. An image of a table
2. A machine-extracted HTML version (from UniTable OCR)

The HTML structure (rows, columns, spanning cells) is generally CORRECT.
But cell text may have OCR errors: wrong characters, broken decimals,
missing symbols, garbled special characters.

Return a corrected JSON table: {"headers": ["col1", ...], "rows": [["val1", ...], ...]}

RULES:
- KEEP the exact same structure (same rows and columns) as the HTML
- FIX cell text by comparing the HTML to what you see in the image
- Preserve text exactly as shown in the image (leading zeros, punctuation, symbols)
- Empty cells → ""
- Do NOT add, remove, or reorder rows/columns
"""

print(f"Gemini client ready (model: {GEMINI_MODEL})")

In [ ]:
# ============================================================
# Cell 4: Hybrid extraction functions
# ============================================================
import json
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup as bs


def html_to_dataframe(html_str: str) -> pd.DataFrame:
    """Convert HTML table string to DataFrame."""
    dfs = pd.read_html(StringIO(html_str))
    if not dfs:
        return pd.DataFrame()
    df = dfs[0]
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [' '.join(str(c) for c in col).strip() for col in df.columns]
    return df


def extract_unitable_only(image: Image.Image) -> pd.DataFrame:
    """Baseline: UniTable extraction only."""
    html = extractor.extract_html(image)
    return html_to_dataframe(html)


def extract_hybrid(image: Image.Image) -> pd.DataFrame:
    """Hybrid: UniTable structure + Gemini refinement."""
    # Step 1: UniTable extracts HTML
    html = extractor.extract_html(image)

    # Step 2: Gemini refines cell content using both image + HTML
    try:
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                REFINE_PROMPT,
                f"\nHere is the machine-extracted HTML:\n```html\n{html}\n```",
                image,
            ],
            config={
                "response_mime_type": "application/json",
            },
        )
        result = json.loads(response.text)
        headers = result.get("headers", [])
        rows = result.get("rows", [])
        if headers and rows:
            return pd.DataFrame(rows, columns=headers)
        elif rows:
            return pd.DataFrame(rows)
    except Exception as e:
        print(f"    Gemini refinement failed ({e}), falling back to UniTable-only")

    # Fallback: use UniTable HTML directly
    return html_to_dataframe(html)


def extract_gemini_only(image: Image.Image) -> pd.DataFrame:
    """Baseline: Gemini-only extraction (no UniTable)."""
    prompt = """Extract the table from this image. Return JSON:
{"headers": ["col1", ...], "rows": [["val1", ...], ...]}
Preserve all text exactly. Empty cells → "". Each row must match header count."""
    try:
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[prompt, image],
            config={"response_mime_type": "application/json"},
        )
        result = json.loads(response.text)
        headers = result.get("headers", [])
        rows = result.get("rows", [])
        if headers and rows:
            return pd.DataFrame(rows, columns=headers)
        elif rows:
            return pd.DataFrame(rows)
    except Exception as e:
        print(f"    Gemini-only extraction failed: {e}")
    return pd.DataFrame()


print("Extraction functions defined")

In [ ]:
# ============================================================
# Cell 5: Evaluation utilities
# ============================================================

def normalize_text(text) -> str:
    """Normalize for comparison."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace('\u00b7', '.').replace('\u2013', '-').replace('\u2212', '-')
    text = text.replace(",", "").replace(".", "").lower()
    return " ".join(text.split())


def compare_tables(df_truth: pd.DataFrame, df_extracted: pd.DataFrame) -> dict:
    """Positional cell-level comparison."""
    truth_rows = len(df_truth)
    extracted_rows = len(df_extracted)
    compare_rows = min(truth_rows, extracted_rows)
    compare_cols = min(len(df_truth.columns), len(df_extracted.columns))

    total_cells = 0
    correct_cells = 0
    rows_fully_correct = 0

    for row_idx in range(compare_rows):
        row_ok = True
        for col_idx in range(compare_cols):
            total_cells += 1
            if normalize_text(df_truth.iloc[row_idx, col_idx]) == normalize_text(df_extracted.iloc[row_idx, col_idx]):
                correct_cells += 1
            else:
                row_ok = False
        if row_ok:
            rows_fully_correct += 1

    return {
        "truth_rows": truth_rows,
        "extracted_rows": extracted_rows,
        "rows_compared": compare_rows,
        "total_cells": total_cells,
        "correct_cells": correct_cells,
        "cell_accuracy": correct_cells / total_cells if total_cells else 0,
        "rows_fully_correct": rows_fully_correct,
        "row_match": rows_fully_correct / compare_rows if compare_rows else 0,
    }


print("Eval utilities ready")

In [ ]:
# ============================================================
# Cell 6: Load PubTabNet ground truth
# ============================================================
import json, re
from io import StringIO

JSONL_PATH = os.path.join(EXAMPLES_DIR, "PubTabNet_Examples.jsonl")

def format_gt_html(ex):
    """Build ground truth HTML from PubTabNet JSONL annotation."""
    html_string = '<html><body><table>%s</table></body></html>' % ''.join(
        ex['html']['structure']['tokens'])
    cell_nodes = list(re.finditer(r'(<td[^<>]*>)(</td>)', html_string))
    cells = [''.join(c['tokens']) for c in ex['html']['cells']]
    assert len(cell_nodes) == len(cells)
    offset = 0
    for n, cell in zip(cell_nodes, cells):
        html_string = html_string[:n.end(1)+offset] + cell + html_string[n.start(2)+offset:]
        offset += len(cell)
    return html_string

examples = [json.loads(line) for line in open(JSONL_PATH)]

ground_truths = {}
for ex in examples:
    fname = ex["filename"]
    try:
        html_str = format_gt_html(ex)
        dfs = pd.read_html(StringIO(html_str))
        if dfs:
            df = dfs[0]
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [' '.join(str(c) for c in col).strip() for col in df.columns]
            ground_truths[fname] = df
    except Exception as e:
        print(f"  Skip {fname}: {e}")

print(f"Ground truths: {len(ground_truths)}/{len(examples)}")

In [ ]:
# ============================================================
# Cell 7: Run 3-way comparison — UniTable vs Gemini vs Hybrid
# ============================================================
import time
from IPython.display import display, HTML

methods = {
    "unitable": extract_unitable_only,
    "gemini": extract_gemini_only,
    "hybrid": extract_hybrid,
}

all_results = {m: [] for m in methods}
timings = {m: [] for m in methods}

for fname, df_truth in ground_truths.items():
    img_path = os.path.join(EXAMPLES_DIR, fname)
    if not os.path.exists(img_path):
        continue

    print(f"\n{'='*70}")
    print(f"{fname} — GT: {len(df_truth)} rows x {len(df_truth.columns)} cols")
    print(f"{'='*70}")

    image = Image.open(img_path).convert("RGB")

    for method_name, extract_fn in methods.items():
        t0 = time.time()
        try:
            df_ext = extract_fn(image)
        except Exception as e:
            print(f"  {method_name}: FAILED — {e}")
            all_results[method_name].append({"total_cells": 0, "correct_cells": 0})
            continue
        elapsed = time.time() - t0
        timings[method_name].append(elapsed)

        if df_ext.empty:
            print(f"  {method_name}: empty result ({elapsed:.1f}s)")
            all_results[method_name].append({"total_cells": 0, "correct_cells": 0})
            continue

        result = compare_tables(df_truth, df_ext)
        all_results[method_name].append(result)
        print(f"  {method_name}: {result['correct_cells']}/{result['total_cells']} cells "
              f"({result['cell_accuracy']:.1%}), "
              f"{result['rows_fully_correct']}/{result['rows_compared']} rows "
              f"({result['row_match']:.1%}) — {elapsed:.1f}s")

In [ ]:
# ============================================================
# Cell 8: Aggregate results — summary table
# ============================================================
import numpy as np

summary_rows = []

for method_name in methods:
    results = all_results[method_name]
    evaluated = [r for r in results if r.get("total_cells", 0) > 0]

    if not evaluated:
        summary_rows.append({
            "Method": method_name,
            "Tables Evaluated": 0,
            "Cell Accuracy": "N/A",
            "Row Match": "N/A",
            "Avg Time (s)": "N/A",
        })
        continue

    total_cells = sum(r["total_cells"] for r in evaluated)
    correct_cells = sum(r["correct_cells"] for r in evaluated)
    total_rows = sum(r["rows_compared"] for r in evaluated)
    correct_rows = sum(r["rows_fully_correct"] for r in evaluated)
    avg_time = np.mean(timings[method_name]) if timings[method_name] else 0

    summary_rows.append({
        "Method": method_name,
        "Tables Evaluated": len(evaluated),
        "Cell Accuracy": f"{correct_cells}/{total_cells} ({correct_cells/total_cells:.1%})",
        "Row Match": f"{correct_rows}/{total_rows} ({correct_rows/total_rows:.1%})",
        "Avg Time (s)": f"{avg_time:.1f}",
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 70)
print("AGGREGATE RESULTS: UniTable vs Gemini vs Hybrid")
print("=" * 70)
display(summary_df)

In [ ]:
# ============================================================
# Cell 9: Per-table breakdown with visual comparison
# ============================================================

# Show first 5 tables side-by-side for the best and worst hybrid results
hybrid_results = all_results["hybrid"]
fnames = [fname for fname in ground_truths if os.path.exists(os.path.join(EXAMPLES_DIR, fname))]

# Build per-table comparison
per_table = []
for i, fname in enumerate(fnames):
    row = {"filename": fname}
    for method_name in methods:
        r = all_results[method_name][i] if i < len(all_results[method_name]) else {}
        acc = r.get("cell_accuracy", 0)
        row[f"{method_name}_cell_acc"] = f"{acc:.1%}" if r.get("total_cells", 0) > 0 else "FAIL"
    per_table.append(row)

per_table_df = pd.DataFrame(per_table)
print("Per-table cell accuracy:")
display(per_table_df)